<a href="https://colab.research.google.com/github/mehrerm/TFM/blob/main/notebooks/carga_depuracion_cervix_csv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



En esta aplicación implementaremos un modelo de puntuación de riesgo poblacional a la mortalidad debido a los cánceres más comunes y su relación con las cantidades y tecnologías usadas en radioterapia por país.

In [1]:
#Cargo o importo pandas, numpy, Matplotlib,
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns
import statsmodels.api as sm




import requests
import unicodedata
import os
from pathlib import Path

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, roc_curve

!pip install optbinning
from optbinning import OptimalBinning, Scorecard, BinningProcess
from sklearn.linear_model import LogisticRegression
from urllib.parse import quote




#lugar donde se van guardando las figuras

output_path = Path("TFM/data/processed")
output_path.mkdir(parents=True, exist_ok=True)

fig_dir = "figuras"
os.makedirs(fig_dir, exist_ok=True)

#########################

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.8/214.8 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.1/28.1 MB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.8/302.8 kB 17.0 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grain 0.2.15 requires protobuf>=5.28.3, but you have protobuf 5.26.1 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 5.26.1 which is incompatib

# Carga, Exploración y Preparación de los datos sobre equipos de radioterapia y pacientes
Se utilizará una base de datos de instalaciones de equipos para radioterapia registrados DIRAC de la IAEA junto con los de la OMS, son datos reales, se buscará el año más actual posible con una cantidad de datos suficientes para hacer el estudio y que tenga lo menos posible la influencia del COVID-19, así como un año anterior a éste.


Los datos de las instalaciones de radioterapia pueden descargarse en la página de la IAEA en el apartado de DIRAC https://dirac.iaea.org/Query/Countries  donde aparecen reflejados los datos de los equipos de radioterapia y el año de sus ultimas actualizaciones a nivel the hardware, por otro lado, se descargaron los datos de la OMS, en el Global Cancer Observatory, donde se descargaron tanto las incidencias como las mortalidades por cáncer y sexo a nivel mundial https://gco.iarc.fr/overtime/en/dataviz/trends?populations=752&sexes=1_2&types=1&multiple_populations=1.
Asimismo, se descargó la información demográfica encontrada en la ONU en el apartado "United Nations World Population Prospects (WPP) para poder ver la densidad poblacional.
Por otro lado, también se ha visto que es necesario para entender mejor la calidad de vida y sobreviviencia, que el PIB per capita puede jugar un papel importante, por lo tanto, también se sumará ese valor a los paises a estudiar, en ese caso se usará una url directa que da los datos sin necesidad de guardarlos en el repositorio que se tiene para tal fin.

## Carga de datos

In [2]:
# Cargamos los datos

# URL base del repositorio
url_base = "https://raw.githubusercontent.com/mehrerm/TFM/main/data/raw/"

#DIRAC
#Datos de los equipos y centros de radioterapia por país

url_dirac = (
    url_base + "DIRAC_Countries.xlsx"
)

dirac = pd.read_excel(url_dirac)
print(["Para DIRAC se tiene:"])
print(dirac.head(5))

#GLOBOCAN
#datos de tipos de cancer más tratados con radioterapia, incidencia, mortalidad
#por año
# Archivos
files_cancer = {
    "lung": "dataset-asr-inc-and-mort-males-and-females-lung.csv",
    "breast": "dataset-asr-inc-and-mort-males-and-females-breast.csv",
    "prostate": "dataset-asr-inc-and-mort-males-and-females-prostat.csv",
    #"colon": "dataset-asr-inc-and-mort-males-and-females-colon.csv",
    "cervix": "dataset-asr-inc-and-mort-males-and-females-cervix-uterino.csv",
    "lip_oral": "dataset-asr-inc-and-mort-males-and-females-lip-oral-cavity-and-pharynx.csv"
    "esofagus": "dataset-asr-inc-and-mort-males-and-females-esophagus.csv
}

#Delaney et al., 2005

#Delaney, G., Jacob, S., Featherstone, C., & Barton, M.
#The role of radiotherapy in cancer treatment: estimating optimal utilization
#from a review of evidence-based clinical guidelines.
#Cancer, 2005.
##########################################################
# Carga de datasets
dfd_cancer = []
print(["Para GLOBOCAN se tiene:"])
for cancer, filename in files_cancer.items():
    url = url_base + quote(filename)
    df_cancer = pd.read_csv(url)



    dfd_cancer.append(df_cancer)

    print(f"{cancer.upper():10s} -> shape: {df_cancer.shape}")

df_all_cancer = pd.concat(dfd_cancer, ignore_index=True)


#ONU
#Población mundial acorde con los datos de la ONU

url_pop = (
       url_base + "WPP2024_TotalPopulationBySex.csv.gz"
)


pop = pd.read_csv(url_pop, compression='gzip')
print(["Para la ONU se tiene:"])
print(pop.head(5))


# Banco mundial para estudiar los pib per capita
url_BM = (
    "https://api.worldbank.org/v2/country/all/indicator/NY.GDP.PCAP.CD"
    "?format=json&per_page=20000"
)



PIB_ = requests.get(url_BM).json()

df_PIB_raw = pd.DataFrame(PIB_[1])

df_PIB = df_PIB_raw[["countryiso3code", "country", "date", "value", "indicator"]].copy()
print(["Para el banco mundial se tiene"])
print(df_PIB.head(5))




['Para DIRAC se tiene:']
     Country      Region Name  RTCenters With RT  \
0    Albania  Southern Europe                  3   
1    Algeria  Northern Africa                 15   
2     Angola    Middle Africa                  2   
3  Argentina    South America                 89   
4    Armenia     Western Asia                  2   

   He Photon And Electron Beam Rt  Proton Ion Therapy  XRay Generator  \
0                               5                   0               1   
1                              37                   0               0   
2                               3                   0               0   
3                             131                   0               8   
4                               5                   0               0   

   Brachy Therapy Inc El Last Update  
0                      0        2025  
1                      9        2023  
2                      1        2023  
3                     47        2025  
4                      3    

/tmp/ipython-input-469489902.py:62: DtypeWarning: Columns (2,3,4,7) have mixed types. Specify dtype option on import or set low_memory=False.
  pop = pd.read_csv(url_pop, compression='gzip')


['Para la ONU se tiene:']
   SortOrder  LocID Notes ISO3_code ISO2_code  SDMX_code  LocTypeID  \
0        NaN   5507   NaN       NaN       NaN        NaN        NaN   
1        NaN   5507   NaN       NaN       NaN        NaN        NaN   
2        NaN   5507   NaN       NaN       NaN        NaN        NaN   
3        NaN   5507   NaN       NaN       NaN        NaN        NaN   
4        NaN   5507   NaN       NaN       NaN        NaN        NaN   

  LocTypeName  ParentID                           Location  VarID Variant  \
0         NaN       NaN  ADB region: Central and West Asia      2  Medium   
1         NaN       NaN  ADB region: Central and West Asia      2  Medium   
2         NaN       NaN  ADB region: Central and West Asia      2  Medium   
3         NaN       NaN  ADB region: Central and West Asia      2  Medium   
4         NaN       NaN  ADB region: Central and West Asia      2  Medium   

   Time  MidPeriod    PopMale  PopFemale   PopTotal  PopDensity  
0  1950     1950.5

## Descripción inicial de los datos

## Depuración y exploración de los datos

### ONU
Con esta dataset, se pretende conseguir la población global por año

In [3]:
print(pop.head())
print(pop["LocTypeName"].unique())

   SortOrder  LocID Notes ISO3_code ISO2_code  SDMX_code  LocTypeID  \
0        NaN   5507   NaN       NaN       NaN        NaN        NaN   
1        NaN   5507   NaN       NaN       NaN        NaN        NaN   
2        NaN   5507   NaN       NaN       NaN        NaN        NaN   
3        NaN   5507   NaN       NaN       NaN        NaN        NaN   
4        NaN   5507   NaN       NaN       NaN        NaN        NaN   

  LocTypeName  ParentID                           Location  VarID Variant  \
0         NaN       NaN  ADB region: Central and West Asia      2  Medium   
1         NaN       NaN  ADB region: Central and West Asia      2  Medium   
2         NaN       NaN  ADB region: Central and West Asia      2  Medium   
3         NaN       NaN  ADB region: Central and West Asia      2  Medium   
4         NaN       NaN  ADB region: Central and West Asia      2  Medium   

   Time  MidPeriod    PopMale  PopFemale   PopTotal  PopDensity  
0  1950     1950.5  35880.164  33333.260  69

In [4]:
# En LocTypeName, filtrar: países, escenario Medium
ONU_c = pop[
    (pop["LocTypeName"] == "Country/Area") &
    (pop["Variant"] == "Medium")
    #El escenario es Medium porque es el escenario más estandarizado
    #esto se debe a que también presenta datos extrapolados sergún distintos
    #criterios
   # (pop["Time"].isin([2016, 2022]))

].copy()

# Seleccionar y renombrar columnas
ONU_c = (
    ONU_c[["Location", "Time", "PopTotal", "PopMale","PopFemale"]]
    .rename(columns={
    "Location": "Country_harmonized",
    "Time": "Year",
    "PopTotal": "Population",
    "PopMale": "Population_male",
    "PopFemale": "Population_female"
})
)

# Convertir tipos
ONU_c["Year"] = ONU_c["Year"].astype(int)

# PopTotal está en miles, pasar a personas
cols = ["Population", "Population_male", "Population_female"]
ONU_c[cols] = ONU_c[cols] * 1000

# Comprobaciones rápidas
print(ONU_c.shape)
print(ONU_c.head(5))

# Guardar CSV reducido
ONU_c.to_csv(
    "population_UN_WPP2024.csv",
    index=False
)

print(ONU_c["Country_harmonized"].unique())
ONU_c["Year"].unique()

(35787, 5)
       Country_harmonized  Year  Population  Population_male  \
336230            Burundi  1950   2254938.0        1080184.0   
336231            Burundi  1951   2305746.0        1105816.0   
336232            Burundi  1952   2355804.0        1130995.0   
336233            Burundi  1953   2405186.0        1155833.0   
336234            Burundi  1954   2454586.0        1180690.0   

        Population_female  
336230          1174755.0  
336231          1199930.0  
336232          1224809.0  
336233          1249353.0  
336234          1273896.0  
['Burundi' 'Comoros' 'Djibouti' 'Eritrea' 'Ethiopia' 'Kenya' 'Madagascar'
 'Malawi' 'Mauritius' 'Mayotte' 'Mozambique' 'Réunion' 'Rwanda'
 'Seychelles' 'Somalia' 'South Sudan' 'Uganda'
 'United Republic of Tanzania' 'Zambia' 'Zimbabwe' 'Angola' 'Cameroon'
 'Central African Republic' 'Chad' 'Congo'
 'Democratic Republic of the Congo' 'Equatorial Guinea' 'Gabon'
 'Sao Tome and Principe' 'Algeria' 'Egypt' 'Libya' 'Morocco' 'Sudan'
 'Tu

array([1950, 1951, 1952, 1953, 1954, 1955, 1956, 1957, 1958, 1959, 1960,
       1961, 1962, 1963, 1964, 1965, 1966, 1967, 1968, 1969, 1970, 1971,
       1972, 1973, 1974, 1975, 1976, 1977, 1978, 1979, 1980, 1981, 1982,
       1983, 1984, 1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993,
       1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004,
       2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015,
       2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026,
       2027, 2028, 2029, 2030, 2031, 2032, 2033, 2034, 2035, 2036, 2037,
       2038, 2039, 2040, 2041, 2042, 2043, 2044, 2045, 2046, 2047, 2048,
       2049, 2050, 2051, 2052, 2053, 2054, 2055, 2056, 2057, 2058, 2059,
       2060, 2061, 2062, 2063, 2064, 2065, 2066, 2067, 2068, 2069, 2070,
       2071, 2072, 2073, 2074, 2075, 2076, 2077, 2078, 2079, 2080, 2081,
       2082, 2083, 2084, 2085, 2086, 2087, 2088, 2089, 2090, 2091, 2092,
       2093, 2094, 2095, 2096, 2097, 2098, 2099, 21

In [5]:
#nombres de paises, se estandarizarán los nombres de los paises de todos los
#dataset

country_mapping = {

    "Macau, China": "Macao",
    "Taiwan, China": "Taiwan",
    'China, Hong Kong SAR': "Hong Kong",
    "China, Macao SAR": "Macao",
    'China, Taiwan Province of China': "Taiwan",
    'China, Republic of China': "China",
    "Macao SAR, China": "Macao",

    'Lao PDR': "Lao",
    "Lao People's Democratic Republic": "Lao",



    'Czech Republic': 'Czechia',

    "Slovak Republic": "Slovakia",

    "United Kingdom of Great Britain and Northern Ireland": "United Kingdom",

    "UK, England": "United Kingdom",
    "UK, Wales": "United Kingdom",
    "UK, Scotland": "United Kingdom",
    "UK, Northern Ireland": "United Kingdom",
    "UK, England and wales": "United Kingdom",


    "France (metropolitan)": "France",
    "France, Martinique": "Martinique",

    "USA": "United States of America",
    "United States" : "United States of America",
    "United States of America (the)": "United States of America",

    "Puerto Rico (US)": "Puerto Rico",



    "Korea, Republic of": "South Korea",
    "Republic of Korea" : "South Korea",
    "Korea, Rep.": "South Korea",
    "Korea, Democratic People's Republic of": "North Korea",


    "Iran, Islamic Republic of": "Iran",
    "Iran (Islamic Republic of)": "Iran",

    "Viet Nam": "Vietnam",

    'Venezuela, Bolivarian Republic of': 'Venezuela',
    'Venezuela (Bolivarian Republic of)': 'Venezuela',
    "Venezuela, RB": "Venezuela",

    'Bolivia, Plurinational State of': 'Bolivia',
    'Bolivia (Plurinational State of)': 'Bolivia',

    "Netherlands, Kingdom of the": "The Netherlands",

    "Republic of Moldova": "Moldova", # Comma added here

    "Kyrgyz Republic": "Kyrgyzstan",

    "Syrian Arab Republic": "Syria"

}

En la celda anterior presenta una lista de países cuyos nombres se han cambiado para que coincidan los 4 datasets.

In [6]:
# se limpian los caracteres de los paises y se cambian los nombres de aquellos
#que sean necesarios


ONU_c["Country_harmonized"] = ONU_c["Country_harmonized"].replace(country_mapping)

ONU_c["Country_harmonized"] = ONU_c["Country_harmonized"].apply(
    lambda x: unicodedata.normalize("NFKD", x)
        .encode("ASCII", "ignore")
        .decode("utf-8") if pd.notna(x) else x
)

print(ONU_c['Country_harmonized'].unique())
print(ONU_c.info())
print(ONU_c.isna().sum().sum())
print(ONU_c['Year'].unique())


['Burundi' 'Comoros' 'Djibouti' 'Eritrea' 'Ethiopia' 'Kenya' 'Madagascar'
 'Malawi' 'Mauritius' 'Mayotte' 'Mozambique' 'Reunion' 'Rwanda'
 'Seychelles' 'Somalia' 'South Sudan' 'Uganda'
 'United Republic of Tanzania' 'Zambia' 'Zimbabwe' 'Angola' 'Cameroon'
 'Central African Republic' 'Chad' 'Congo'
 'Democratic Republic of the Congo' 'Equatorial Guinea' 'Gabon'
 'Sao Tome and Principe' 'Algeria' 'Egypt' 'Libya' 'Morocco' 'Sudan'
 'Tunisia' 'Western Sahara' 'Botswana' 'Eswatini' 'Lesotho' 'Namibia'
 'South Africa' 'Benin' 'Burkina Faso' 'Cabo Verde' "Cote d'Ivoire"
 'Gambia' 'Ghana' 'Guinea' 'Guinea-Bissau' 'Liberia' 'Mali' 'Mauritania'
 'Niger' 'Nigeria' 'Saint Helena' 'Senegal' 'Sierra Leone' 'Togo'
 'Kazakhstan' 'Kyrgyzstan' 'Tajikistan' 'Turkmenistan' 'Uzbekistan'
 'China' 'Hong Kong' 'Macao' 'Taiwan' "Dem. People's Republic of Korea"
 'Japan' 'Mongolia' 'South Korea' 'Afghanistan' 'Bangladesh' 'Bhutan'
 'India' 'Iran' 'Maldives' 'Nepal' 'Pakistan' 'Sri Lanka'
 'Brunei Darussalam' 'C

Variables a utilizar:

* Country_harmonized: los países con información

* Year: año

* Population: población tanto hombres como mujeres

* Population_male: pobración de hombres

* Population_female: población de mujeres

Los datos de población utilizados en este estudio proceden de la base World Population Prospects 2024 elaborada por la División de Población del Departamento de Asuntos Económicos y Sociales de las Naciones Unidas (ONU). Según la metodología oficial de esta fuente, las estimaciones de población se basan en datos observados provenientes de censos nacionales, registros vitales y encuestas demográficas, mientras que las proyecciones de población comienzan a partir del año 2024.

Aunque el conjunto de datos incluye estimaciones hasta el año 2023, la disponibilidad y calidad de los datos demográficos recientes varía considerablemente entre países. En muchos casos, los últimos censos o registros vitales utilizados como base empírica corresponden a años anteriores, especialmente en el periodo posterior a la pandemia de COVID-19. Como consecuencia, los valores más recientes incorporan un mayor grado de interpolación y ajuste modelizado.

Por este motivo, y con el objetivo de evitar el uso de datos proyectados o altamente modelizados, y debido a las exploraciones posteriores con el resto de nuestras fuentes de datos, en este trabajo se selecciona el año 2022 como el año más reciente con una cobertura amplia y consistente de datos poblacionales basados mayoritariamente en información observada (y no extrapolada).

La utilización del año 2022 resulta, además, especialmente adecuada para su integración con los datos de mortalidad por cáncer de GLOBOCAN y los datos de infraestructura de radioterapia del registro DIRAC, que presentan una mayor disponibilidad y estabilidad en torno a ese periodo temporal. De este modo, la fusión de las fuentes se realiza minimizando sesgos del uso de proyecciones demográficas.

Asimismo, se utilizará el año 2016, ya que es un año también bastante consistente donde además, en GLOBOCAN, presenta buena recolección de datos sobre incidencias, algo que más adelante se demostrará que años posteriores, este valor estará ausente a la mayoría de países.


#DIRAC
para conocer los centros de radioterapia y sus respectivos equipamientos

In [7]:
dirac.head()



,Country,Region Name,RTCenters With RT,He Photon And Electron Beam Rt,Proton Ion Therapy,XRay Generator,Brachy Therapy Inc El,Last Update
0,Albania,Southern Europe,3,5,0,1,0,2025
1,Algeria,Northern Africa,15,37,0,0,9,2023
2,Angola,Middle Africa,2,3,0,0,1,2023
3,Argentina,South America,89,131,0,8,47,2025
4,Armenia,Western Asia,2,5,0,0,3,2025


In [8]:
dirac.shape

(156, 8)

In [9]:
dirac.columns


Index(['Country', 'Region Name', 'RTCenters With RT',
       'He Photon And Electron Beam Rt', 'Proton Ion Therapy',
       'XRay Generator', 'Brachy Therapy Inc El', 'Last Update'],
      dtype='object')

In [10]:
# Seleccionar y renombrar columnas
dirac = (
    dirac[['Country', 'Region Name', 'RTCenters With RT',
       'He Photon And Electron Beam Rt', 'Proton Ion Therapy',
       'XRay Generator', 'Brachy Therapy Inc El', 'Last Update']]
    .rename(columns={
    'RTCenters With RT': "RTCenters",
    'He Photon And Electron Beam Rt': "Linac",
    'Proton Ion Therapy': "Protontherapy",
    'XRay Generator': "XRay",
    'Brachy Therapy Inc El': "Brachytherapy"
})
)

In [11]:
dirac.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 156 entries, 0 to 155
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Country        155 non-null    object
 1   Region Name    155 non-null    object
 2   RTCenters      156 non-null    int64 
 3   Linac          156 non-null    int64 
 4   Protontherapy  156 non-null    int64 
 5   XRay           156 non-null    int64 
 6   Brachytherapy  156 non-null    int64 
 7   Last Update    156 non-null    object
dtypes: int64(5), object(3)
memory usage: 9.9+ KB


##Variables encontradas
Entonces, con estos datos se tiene lo siguiente:

* 'Country_harmonized' = país al que corresponde la información

* 'Region Name' = región geográfica o continental en la que se localiza el país.

* 'RTCenters' = número de centros que disponen de al menos un equipo de radioterapia.

* 'Linacs' = número de centros con equipos de teleterapia que utilizan haces de fotones y electrones.

* 'Protontherapy'= número de centros con disponibilidad de protonterapia.

* 'XRay' = número de generadores de rayos X utilizados con fines terapéuticos.

* 'Brachytherapy' = número de equipos de braquiterapia, incluyendo tanto fuentes radiactivas como sistemas de braquiterapia electrónica mediante generadores miniaturizados de rayos X.

* 'Last Update' = año de la última actualización de la información registrada para cada país.

Las variables pueden clasificarse en cualitativas nominales, cuantitativas discretas y una variable temporal, lo que permite realizar posteriormente un análisis exploratorio orientado a la detección de valores atípicos (outliers) y a la normalización por población.

Ahora se realiza un análisis exploratorio para buscar outliers, si es que los hay.




In [12]:
categ_vars = dirac.select_dtypes(include='object').columns


In [13]:
#explotar las variables categoricas

for col in categ_vars:
    print(f"Valores únicos en {col}:")
    print(dirac[col].unique())
    print("-" * 40)

Valores únicos en Country:
['Albania' 'Algeria' 'Angola' 'Argentina' 'Armenia' 'Aruba' 'Australia'
 'Austria' 'Azerbaijan' 'Bahamas' 'Bahrain' 'Bangladesh' 'Barbados'
 'Belarus' 'Belgium' 'Bermuda' 'Bolivia, Plurinational State of'
 'Bosnia and Herzegovina' 'Botswana' 'Brazil' 'Brunei Darussalam'
 'Bulgaria' 'Burkina Faso' 'Cambodia' 'Cameroon' 'Canada' 'Chile' 'China'
 'Colombia' 'Costa Rica' 'Croatia' 'Cuba' 'Cyprus' 'Czech Republic'
 "Côte d'Ivoire" 'Dem. P.R. of Korea' 'Democratic Republic of the Congo'
 'Denmark' 'Dominican Republic' 'Ecuador' 'Egypt' 'El Salvador' 'Estonia'
 'Ethiopia' 'Finland' 'France' 'Gabon' 'Georgia' 'Germany' 'Ghana'
 'Greece' 'Guadeloupe' 'Guatemala' 'Guyana' 'Honduras' 'Hungary' 'Iceland'
 'India' 'Indonesia' 'Iran, Islamic Republic of' 'Iraq' 'Ireland' 'Israel'
 'Italy' 'Jamaica' 'Japan' 'Jordan' 'Kazakhstan' 'Kenya'
 'Korea, Republic of' 'Kuwait' 'Kyrgyzstan'
 "Lao People's Democratic Republic" 'Latvia' 'Lebanon' 'Libya' 'Lithuania'
 'Luxembourg' 'Macau

En este primera análisis exploratorio, parece que existen unos valores anómalos, se explora entonces qué información existe cuando se trata de "nan" y "Latest: 2025"

In [14]:
dirac[dirac['Last Update'] == 'Latest: 2025']





,Country,Region Name,RTCenters,Linac,Protontherapy,XRay,Brachytherapy,Last Update
155,NaN,NaN,8620,17162,133,706,3359,Latest: 2025


Esto parece más bien que es la última fila con todos los equipos.

In [15]:
dirac['Country'].isna().sum()

np.int64(1)

Con esto se demuestra que estos valores anómalos están en la última fila por consecuencia de sumar todos los equipos y centros y calcular la última actualización, entonces, esta fila se eliminará de los datos. Otra operación que sería interesante modificar son los nombres con caracteres que nos podría dar problemas, así como asegurarse de que no hay paises repetidos.

In [16]:
dirac = dirac.drop(index=155)
#queda eliminada la ultima fila
#dirac = dirac.drop(columns=["Region name"], errors="ignore")
#Ya que el análisis será por país, ya luego se verá si la región tiene sentido.

dirac['Country'].value_counts().sort_values(ascending=False)


,count
Country,
Albania,1
Algeria,1
Angola,1
Argentina,1
Armenia,1
...,...
"Venezuela, Bolivarian Republic of",1
Viet Nam,1
Yemen,1


In [17]:
#se reemplazan los nombres de algunos paises

dirac["Country_harmonized"] = dirac["Country"].replace(country_mapping)

dirac["Country_harmonized"] = dirac["Country_harmonized"].apply(
    lambda x: unicodedata.normalize("NFKD", x)
        .encode("ASCII", "ignore")
        .decode("utf-8") if pd.notna(x) else x
)


Debido a que hay países que están segregados por zonas, se unificarán usando Country_harmonized.

Durante el proceso de armonización geográfica se identificaron territorios no soberanos, como Martinica, que inicialmente se consideraron candidatos a ser integrados bajo el Estado correspondiente (Francia). No obstante, un análisis exploratorio de los datos epidemiológicos reveló diferencias sustanciales entre Francia y Martinica en las variables clave de GLOBOCAN, tales como la tasa estandarizada por edad (ASR World), la tasa bruta, el riesgo acumulado y el número total de casos. Por lo tanto, se usarán por separado, algo que no ocurre con UK, donde todo el territorio tiene resultados estadísticos similares, donde entonces si se sumarán casos y se promediarán los datos numeros como el ASR que son tasas.

Adicionalmente, se constató que ambas fuentes de datos GLOBOCAN y DIRAC, proporcionan información diferenciada para Martinica y Francia. En consecuencia, y con el fin de preservar la coherencia interna de los datos y evitar la introducción de sesgos derivados de agregaciones no justificadas, se decidió mantener Martinica como una entidad separada en el análisis.


In [18]:
# Comprobar duplicados por país
n_dups = dirac["Country_harmonized"].duplicated().sum()

if n_dups > 0:
    num_cols = dirac.select_dtypes(include="number").columns.tolist()

    # Last Update NO se suma, se queda el más reciente
    if "Last Update" in num_cols:
        num_cols.remove("Last Update")

    dirac_clean = (
        dirac
        .groupby("Country_harmonized", as_index=False)
        .agg(
            {**{c: "sum" for c in num_cols},
             "Last Update": "max",
             "Region Name": "first"} # Explicitly include Region Name
        )
    )
else:
  dirac_clean = dirac.copy()

dirac_clean["Country_harmonized"].duplicated().sum()

np.int64(0)

Dado que el único campo no numérico del conjunto DIRAC corresponde al año de la última actualización, no fue necesario aplicar reglas de agregación adicionales para variables categóricas.

In [19]:
dirac_clean['Country_harmonized'].unique()

array(['Albania', 'Algeria', 'Angola', 'Argentina', 'Armenia', 'Aruba',
       'Australia', 'Austria', 'Azerbaijan', 'Bahamas', 'Bahrain',
       'Bangladesh', 'Barbados', 'Belarus', 'Belgium', 'Bermuda',
       'Bolivia', 'Bosnia and Herzegovina', 'Botswana', 'Brazil',
       'Brunei Darussalam', 'Bulgaria', 'Burkina Faso', 'Cambodia',
       'Cameroon', 'Canada', 'Chile', 'China', 'Colombia', 'Costa Rica',
       'Croatia', 'Cuba', 'Cyprus', 'Czechia', "Cote d'Ivoire",
       'Dem. P.R. of Korea', 'Democratic Republic of the Congo',
       'Denmark', 'Dominican Republic', 'Ecuador', 'Egypt', 'El Salvador',
       'Estonia', 'Ethiopia', 'Finland', 'France', 'Gabon', 'Georgia',
       'Germany', 'Ghana', 'Greece', 'Guadeloupe', 'Guatemala', 'Guyana',
       'Honduras', 'Hungary', 'Iceland', 'India', 'Indonesia', 'Iran',
       'Iraq', 'Ireland', 'Israel', 'Italy', 'Jamaica', 'Japan', 'Jordan',
       'Kazakhstan', 'Kenya', 'South Korea', 'Kuwait', 'Kyrgyzstan',
       'Lao', 'Latvia', 

Se ha limpiado de caracteres anómalos en países.
Con los datos numéricos entonces, se analiza si existe algun dato anomalo

In [20]:
datos_numericos = dirac_clean.select_dtypes(include=['int64'])

datos_numericos.describe().T



,count,mean,std,min,25%,50%,75%,max
RTCenters,155.0,55.612903,232.630563,1.0,2.0,6.0,26.0,2237.0
Linac,155.0,110.722581,409.807581,1.0,3.0,16.0,57.5,3892.0
Protontherapy,155.0,0.858065,4.279882,0.0,0.0,0.0,0.0,44.0
XRay,155.0,4.554839,16.015878,0.0,0.0,0.0,1.0,124.0
Brachytherapy,155.0,21.670968,66.289713,0.0,1.0,3.0,14.5,638.0


Con estos resultados, se puede saber que hay paises con valores extremos y otros donde escasamente hay un solo equipo. Se puede ver que hay paises donde es cero en equipos mientras que otros tienen hasta 2237 centros con radioterapia, esto tiene sentido si se explora que paises son así como su población, esto se hará más adelante con los datos del BM.


In [21]:
# Lista de variables que representan centros/equipos
columnas_equipos = [ "RTCenters", "Linac", "Protontherapy", "XRay", "Brachytherapy" ]

# Mostrar el Top 5 de paises con su población respectivamente
for col in columnas_equipos:
  print(f"\n Top 5 países por: {col}")
  top5 = dirac_clean[['Country_harmonized', col]].sort_values(by=col, ascending=False).head(5)
  print(top5.to_string(index=False))




 Top 5 países por: RTCenters
      Country_harmonized  RTCenters
United States of America       2237
                   China       1624
                   Japan        759
                   India        464
                 Germany        307

 Top 5 países por: Linac
      Country_harmonized  Linac
United States of America   3892
                   China   2932
                   Japan   1067
                   India    793
                 Germany    587

 Top 5 países por: Protontherapy
      Country_harmonized  Protontherapy
United States of America             44
                   Japan             24
                   China             15
                 Germany              7
          United Kingdom              7

 Top 5 países por: XRay
Country_harmonized  XRay
             China   124
           Germany   113
Russian Federation    79
    United Kingdom    49
            Brazil    39

 Top 5 países por: Brachytherapy
      Country_harmonized  Brachytherapy
United States

In [22]:
dirac_clean.head()

,Country,Region Name,RTCenters,Linac,Protontherapy,XRay,Brachytherapy,Last Update,Country_harmonized
0,Albania,Southern Europe,3,5,0,1,0,2025,Albania
1,Algeria,Northern Africa,15,37,0,0,9,2023,Algeria
2,Angola,Middle Africa,2,3,0,0,1,2023,Angola
3,Argentina,South America,89,131,0,8,47,2025,Argentina
4,Armenia,Western Asia,2,5,0,0,3,2025,Armenia


Con estos resultados es necesario tener entonces los datos demograficos por país, ademas de los datos de cancer.
Ahora, a explorar y depurar los datos provenientes de
#GLOBOCAN.
En donde se podrá encontrar los indice de incidencia y mortalidad segun sexo, tipo de cancer, pais y año.
Para esto, se ha tomado seis tipos de cancer muy comunes: Pulmon, mama, prostata, colon, cervix y leucemia.

In [23]:

df_all_cancer["Cancer label"].unique()


array(['Cervix uteri'], dtype=object)

In [24]:
#se cambia solamente el nombre de Cervix uteri
df_all_cancer["Cancer label"] = df_all_cancer["Cancer label"].replace(
    {"Cervix uteri": "Cervix"}
)

df_all_cancer["Cancer label"] = df_all_cancer["Cancer label"].replace(
    {"Lip,  oral cavity and pharynx ": "Lip_oc_Pharynx"}
)


In [25]:
df_all_cancer["Cancer label"].unique()

array(['Cervix'], dtype=object)

In [26]:
df_all_cancer["Cancer label"].value_counts()

,count
Cancer label,
Cervix,1640


In [27]:
df_all_cancer.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1640 entries, 0 to 1639
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Cancer id        1640 non-null   int64  
 1   Cancer label     1640 non-null   object 
 2   Population id    1640 non-null   int64  
 3   Country label    1640 non-null   object 
 4   Sex              1640 non-null   int64  
 5   Type             1640 non-null   int64  
 6   Year             1640 non-null   int64  
 7   ASR (World)      1640 non-null   float64
 8   Crude rate       1640 non-null   float64
 9   Cumulative risk  1640 non-null   float64
 10  Total            1640 non-null   int64  
dtypes: float64(3), int64(6), object(2)
memory usage: 141.1+ KB


Se puede observar que para un cáncer como el que afecta exclusivamente un sexo de la población , los datos son la mitad (1640 filas) del resto donde se toma en cuenta ambos sexos (3280 filas). En el caso de Estados Unidos, GLOBOCAN proporciona información adicional desagregada por grupos poblacionales (p. ej., “USA: White” y “USA: Black”), además de los datos correspondientes al total nacional. Dado que estas categorías no representan unidades geográficas independientes y que el conjunto de datos incluye la población total, dichas desagregaciones se excluyeron del análisis para evitar duplicidades y distorsiones en los indicadores epidemiológicos.

Los distintos conjuntos de datos correspondientes a cada tipo de cáncer se integraron en un único dataframe, incorporando una variable identificadora del tipo de cáncer para facilitar el análisis conjunto.


In [28]:
df_all_cancer = df_all_cancer.drop(columns=["Cancer id", "Population id"], errors="ignore")




In [29]:

print(df_all_cancer.info())


for col in df_all_cancer:
    print(f"\n {col}")
    print(df_all_cancer[col].unique())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1640 entries, 0 to 1639
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Cancer label     1640 non-null   object 
 1   Country label    1640 non-null   object 
 2   Sex              1640 non-null   int64  
 3   Type             1640 non-null   int64  
 4   Year             1640 non-null   int64  
 5   ASR (World)      1640 non-null   float64
 6   Crude rate       1640 non-null   float64
 7   Cumulative risk  1640 non-null   float64
 8   Total            1640 non-null   int64  
dtypes: float64(3), int64(4), object(2)
memory usage: 115.4+ KB
None

 Cancer label
['Cervix']

 Country label
['Belarus' 'Canada' 'Chile' 'China' 'Colombia' 'Costa Rica' 'Croatia'
 'Cuba' 'Cyprus' 'Czechia' 'Denmark' 'Ecuador' 'Estonia' 'Finland'
 'France (metropolitan)' 'Georgia' 'Germany' 'Greece' 'Argentina'
 'Guatemala' 'Guyana' 'Hungary' 'Iceland' 'India' 'Australia' 'Ireland'
 '

Durante el análisis exploratorio se identificaron diferencias en la nomenclatura de países, incluyendo desagregaciones territoriales y subpoblacionales. Estas inconsistencias se abordaron mediante un proceso de armonización de nombres para permitir la integración con otras fuentes de datos. Es necesaior un nombre único por país.

In [30]:

#aplicando maping para cambiar y armonizar los nombres de paises

df_all_cancer["Country_harmonized"] = (
    df_all_cancer["Country label"]
    .replace(country_mapping)
)

# se normalizan los caracteres
df_all_cancer["Country_harmonized"] = df_all_cancer["Country_harmonized"].apply(
    lambda x: unicodedata.normalize("NFKD", x).encode("ASCII", "ignore").decode("utf-8")
    if pd.notna(x) else x
)


print(len(df_all_cancer["Country_harmonized"].unique()))

for col in df_all_cancer:
    print(f"\n {col}")
    print(df_all_cancer[col].unique())


df_all_cancer = df_all_cancer.drop(columns=["Country label"], errors="ignore")

76

 Cancer label
['Cervix']

 Country label
['Belarus' 'Canada' 'Chile' 'China' 'Colombia' 'Costa Rica' 'Croatia'
 'Cuba' 'Cyprus' 'Czechia' 'Denmark' 'Ecuador' 'Estonia' 'Finland'
 'France (metropolitan)' 'Georgia' 'Germany' 'Greece' 'Argentina'
 'Guatemala' 'Guyana' 'Hungary' 'Iceland' 'India' 'Australia' 'Ireland'
 'Israel' 'Italy' 'Japan' 'Austria' 'Korea, Republic of' 'Kuwait'
 'Kyrgyzstan' 'Latvia' 'Lithuania' 'Luxembourg' 'Malta'
 'France, Martinique' 'Bahrain' 'Mauritius' 'Mexico' 'Moldova' 'Armenia'
 'The Netherlands' 'New Zealand' 'Nicaragua' 'Belgium' 'Norway' 'Panama'
 'Paraguay' 'Philippines' 'Poland' 'Portugal' 'Puerto Rico' 'Qatar'
 'Romania' 'Serbia' 'Singapore' 'Slovakia' 'Slovenia' 'South Africa'
 'Spain' 'Sweden' 'Switzerland' 'Brazil' 'Thailand' 'Türkiye' 'Uganda'
 'United Kingdom' 'UK, England' 'UK, Wales' 'UK, Scotland'
 'UK, Northern Ireland' 'UK, England and wales' 'Belize' 'USA'
 'USA: White' 'USA: Black' 'Uruguay' 'Uzbekistan' 'Venezuela']

 Sex
[2]

 Type
[0

In [31]:
df_all_cancer.columns


Index(['Cancer label', 'Sex', 'Type', 'Year', 'ASR (World)', 'Crude rate',
       'Cumulative risk', 'Total', 'Country_harmonized'],
      dtype='object')

In [32]:
type_map = {0: "Incidence", 1: "Mortality"}
sex_map  = {1: "Male", 2: "Female"}


df_all_cancer["Type"] = df_all_cancer["Type"].map(type_map)
df_all_cancer["Sex"]  = df_all_cancer["Sex"].map(sex_map)


df_all_cancer["Cancer label"].unique()

array(['Cervix'], dtype=object)

In [33]:
key_cols = ["Country_harmonized", "Cancer label", "Sex", "Type", "Year"]
df_all_cancer.duplicated(subset=key_cols).sum()

np.int64(78)

In [34]:
df_all_cancer

,Cancer label,Sex,Type,Year,ASR (World),Crude rate,Cumulative risk,Total,Country_harmonized
0,Cervix,Female,Incidence,2007,12.880989,17.994184,1.285189,931,Belarus
1,Cervix,Female,Incidence,2008,13.089796,18.763663,1.296157,956,Belarus
2,Cervix,Female,Incidence,2009,12.789255,18.367835,1.287709,934,Belarus
3,Cervix,Female,Incidence,2010,12.684533,18.316613,1.270706,930,Belarus
4,Cervix,Female,Incidence,2011,13.305420,18.935117,1.313381,960,Belarus
...,...,...,...,...,...,...,...,...,...
1635,Cervix,Female,Mortality,2012,8.973664,8.901073,0.954095,1321,Venezuela
1636,Cervix,Female,Mortality,2013,9.133123,9.222048,0.970629,1386,Venezuela
1637,Cervix,Female,Mortality,2014,9.922533,10.276557,1.023536,1563,Venezuela
1638,Cervix,Female,Mortality,2015,9.740112,10.176412,1.023116,1565,Venezuela


Los datos oncológicos incluyen tanto incidencia como mortalidad, diferenciadas mediante la variable Type. Para los análisis comparativos entre países se emplean principalmente las tasas estandarizadas por edad (ASR (World)), al permitir comparaciones internacionales independientes de la estructura demográfica.
Las tasas de incidencia y mortalidad se expresan como ASR (World), calculadas mediante la estandarización directa usando la población estándar mundial.

Las variables que nos interesan por ahora, son:

* Cancer label: Nombre del tipo de cáncer analizado.

* Country_harmonized: País al que corresponden los datos.

* Sex: Sexo de la población considerada (1 hombre, 2 mujer).

* Year: Año de referencia del dato epidemiológico.

* Type: Tipo de medida epidemiológica, diferenciando entre incidencia y mortalidad (0 incidencia, 1 mortalidad).

* ASR (World): Tasa estandarizada por edad según la población estándar mundial, expresada por 100.000 habitantes.

* Crude rate: Tasa bruta de incidencia o mortalidad por 100.000 habitantes, no ajustada por edad.

* Cumulative risk: Riesgo acumulado de desarrollar o morir por el cáncer hasta una edad determinada.

* Total:número total de detecciones.

Los datos no relevantes para el estudio que son cancer id y population id, se eliminan.

En el caso del Reino Unido, GLOBOCAN proporciona información desagregada por subdivisiones territoriales. Dado que estas corresponden a una misma entidad nacional y que los valores de las tasas epidemiológicas eran similares entre las distintas subdivisiones, se procedió a integrar dichos registros. Los recuentos absolutos se agregaron mediante suma, mientras que las tasas (ASR World, tasa bruta y riesgo acumulado) se combinaron mediante media simple, dado que no se disponía de información poblacional para realizar una ponderación adecuada.

Entonces, a continuación realizaremos la implementación más adecuada posible, se separa UK del resto, se suma el **TOTAL** y se calcula la media del resto de valores.

In [35]:

uk = df_all_cancer[df_all_cancer["Country_harmonized"] == "United Kingdom"]
rest = df_all_cancer[df_all_cancer["Country_harmonized"] != "United Kingdom"]

uk_agg = (
    uk
    .groupby(
        ["Country_harmonized", "Cancer label", "Sex", "Type", "Year"],
        as_index=False
    )
    .agg({
        "Total": "sum",
        "ASR (World)": "mean",
        "Crude rate": "mean",
        "Cumulative risk": "mean"
    })
)

Con esto se tiene entonces UK unificado con medias en sus valores.
Una vez se ha explorado y corregido los datos de GLOBOCAN, se fusionan los datos que se tienen con lo obtenido con DIRAC y GLOBOCAN para hacer un solo dataset.

Adicionalmente, además de concatenar entonces UK co el resto, es necesario cambiar la momenclatura de sex y type para hacerlo más fácil de manipular e interpretar

In [36]:
# Ahora se reconstruye nuevamente

df_all_cancer_clean = pd.concat([rest, uk_agg], ignore_index=True)


#Con los datos ya limpios, se genera entonces un nuevo dataframe
glob_clean = df_all_cancer_clean.copy()

Para el
#Banco mundial
Se tiene entonces

In [37]:
#Se limpia el dataset del banco mundial

print(df_PIB[["country", "date", "value"]].head())

df_PIB["Country"] = df_PIB["country"].apply(lambda x: x["value"])
df_PIB["Year"] = df_PIB["date"].astype(int)
df_PIB["GDP_per_capita"] = df_PIB["value"]

print(df_PIB[["Country", "Year", "GDP_per_capita"]].head())
print(df_PIB["Country"].unique())


                                             country  date        value
0  {'id': 'ZH', 'value': 'Africa Eastern and Sout...  2024  1615.396356
1  {'id': 'ZH', 'value': 'Africa Eastern and Sout...  2023  1571.449189
2  {'id': 'ZH', 'value': 'Africa Eastern and Sout...  2022  1679.327622
3  {'id': 'ZH', 'value': 'Africa Eastern and Sout...  2021  1562.416175
4  {'id': 'ZH', 'value': 'Africa Eastern and Sout...  2020  1351.591669
                       Country  Year  GDP_per_capita
0  Africa Eastern and Southern  2024     1615.396356
1  Africa Eastern and Southern  2023     1571.449189
2  Africa Eastern and Southern  2022     1679.327622
3  Africa Eastern and Southern  2021     1562.416175
4  Africa Eastern and Southern  2020     1351.591669
['Africa Eastern and Southern' 'Africa Western and Central' 'Arab World'
 'Caribbean small states' 'Central Europe and the Baltics'
 'Early-demographic dividend' 'East Asia & Pacific'
 'East Asia & Pacific (excluding high income)'
 'East Asia & Pacif

In [38]:
df_PIB["Country_harmonized"] = (
    df_PIB["Country"]
    .replace(country_mapping)
)

df_PIB["Country_harmonized"] = df_PIB["Country_harmonized"].apply(
    lambda x: unicodedata.normalize("NFKD", x).encode("ASCII", "ignore").decode("utf-8")
    if pd.notna(x) else x
)
df_PIB["Country_harmonized"].unique()

array(['Africa Eastern and Southern', 'Africa Western and Central',
       'Arab World', 'Caribbean small states',
       'Central Europe and the Baltics', 'Early-demographic dividend',
       'East Asia & Pacific',
       'East Asia & Pacific (excluding high income)',
       'East Asia & Pacific (IDA & IBRD countries)', 'Euro area',
       'Europe & Central Asia',
       'Europe & Central Asia (excluding high income)',
       'Europe & Central Asia (IDA & IBRD countries)', 'European Union',
       'Fragile and conflict affected situations',
       'Heavily indebted poor countries (HIPC)', 'High income',
       'IBRD only', 'IDA & IBRD total', 'IDA blend', 'IDA only',
       'IDA total', 'Late-demographic dividend',
       'Latin America & Caribbean',
       'Latin America & Caribbean (excluding high income)',
       'Latin America & the Caribbean (IDA & IBRD countries)',
       'Least developed countries: UN classification',
       'Low & middle income', 'Low income', 'Lower middle in

In [39]:
df_PIB.head()

,countryiso3code,country,date,value,indicator,Country,Year,GDP_per_capita,Country_harmonized
0,AFE,"{'id': 'ZH', 'value': 'Africa Eastern and Sout...",2024,1615.396356,"{'id': 'NY.GDP.PCAP.CD', 'value': 'GDP per cap...",Africa Eastern and Southern,2024,1615.396356,Africa Eastern and Southern
1,AFE,"{'id': 'ZH', 'value': 'Africa Eastern and Sout...",2023,1571.449189,"{'id': 'NY.GDP.PCAP.CD', 'value': 'GDP per cap...",Africa Eastern and Southern,2023,1571.449189,Africa Eastern and Southern
2,AFE,"{'id': 'ZH', 'value': 'Africa Eastern and Sout...",2022,1679.327622,"{'id': 'NY.GDP.PCAP.CD', 'value': 'GDP per cap...",Africa Eastern and Southern,2022,1679.327622,Africa Eastern and Southern
3,AFE,"{'id': 'ZH', 'value': 'Africa Eastern and Sout...",2021,1562.416175,"{'id': 'NY.GDP.PCAP.CD', 'value': 'GDP per cap...",Africa Eastern and Southern,2021,1562.416175,Africa Eastern and Southern
4,AFE,"{'id': 'ZH', 'value': 'Africa Eastern and Sout...",2020,1351.591669,"{'id': 'NY.GDP.PCAP.CD', 'value': 'GDP per cap...",Africa Eastern and Southern,2020,1351.591669,Africa Eastern and Southern


In [40]:
df_PIB = df_PIB.drop(columns=["country", "date", "value", "Country", "countryiso3code", "indicator"], errors="ignore")
datos_numericos = df_PIB.select_dtypes(include=['int64', "float64"])

datos_numericos.describe().T

,count,mean,std,min,25%,50%,75%,max
Year,17290.0,1992.000000,18.762206,1960.000000,1976.000000,1992.000000,2008.000000,2024.000000
GDP_per_capita,14561.0,8701.887816,17595.339722,11.801322,583.782776,1938.810849,8045.946305,288001.433369


In [41]:
df_PIB["Country_harmonized"].unique()

array(['Africa Eastern and Southern', 'Africa Western and Central',
       'Arab World', 'Caribbean small states',
       'Central Europe and the Baltics', 'Early-demographic dividend',
       'East Asia & Pacific',
       'East Asia & Pacific (excluding high income)',
       'East Asia & Pacific (IDA & IBRD countries)', 'Euro area',
       'Europe & Central Asia',
       'Europe & Central Asia (excluding high income)',
       'Europe & Central Asia (IDA & IBRD countries)', 'European Union',
       'Fragile and conflict affected situations',
       'Heavily indebted poor countries (HIPC)', 'High income',
       'IBRD only', 'IDA & IBRD total', 'IDA blend', 'IDA only',
       'IDA total', 'Late-demographic dividend',
       'Latin America & Caribbean',
       'Latin America & Caribbean (excluding high income)',
       'Latin America & the Caribbean (IDA & IBRD countries)',
       'Least developed countries: UN classification',
       'Low & middle income', 'Low income', 'Lower middle in

In [42]:
print(df_PIB.isna().sum().sort_values(ascending=False))
(df_PIB.isna().mean() * 100).sort_values(ascending=False)




GDP_per_capita        2729
Year                     0
Country_harmonized       0
dtype: int64


,0
GDP_per_capita,15.78369
Year,0.00000
Country_harmonized,0.00000


In [43]:
df_PIB_clean = df_PIB.copy()


In [44]:
df_PIB_clean = df_PIB.dropna(subset=["GDP_per_capita"])

df_PIB_clean.isna().sum()


,0
Year,0
GDP_per_capita,0
Country_harmonized,0


Una vez se tienen los datos limpios, se busca paises en común.

In [45]:
glob_c = set(glob_clean["Country_harmonized"].dropna().unique())
dir_c  = set(dirac_clean["Country_harmonized"].dropna().unique())
pop_c  = set(ONU_c["Country_harmonized"].dropna().unique())
BM_c  = set(df_PIB_clean["Country_harmonized"].dropna().unique())

common_4 = sorted(glob_c & dir_c & pop_c & BM_c)
print("Número de países en común:", len(common_4))



Número de países en común: 71


In [46]:
def clean_country_set(s):
    return {
        c for c in s
        if ":" not in c
    }

glob_countries_clean = clean_country_set(glob_c)
dirac_countries_clean = clean_country_set(dir_c)
onu_countries_clean = clean_country_set(pop_c)
BM_countries_clean = clean_country_set(BM_c)

common_4_clean = (
    glob_countries_clean
    & dirac_countries_clean
    & onu_countries_clean
    & BM_countries_clean
)

print(len(common_4_clean))
for c in sorted(common_4_clean):
    print(c)



71
Argentina
Armenia
Australia
Austria
Bahrain
Belarus
Belgium
Brazil
Canada
Chile
China
Colombia
Costa Rica
Croatia
Cuba
Cyprus
Czechia
Denmark
Ecuador
Estonia
Finland
France
Georgia
Germany
Greece
Guatemala
Guyana
Hungary
Iceland
India
Ireland
Israel
Italy
Japan
Kuwait
Kyrgyzstan
Latvia
Lithuania
Luxembourg
Malta
Mauritius
Mexico
Moldova
New Zealand
Nicaragua
Norway
Panama
Paraguay
Philippines
Poland
Portugal
Puerto Rico
Qatar
Romania
Serbia
Singapore
Slovakia
Slovenia
South Africa
South Korea
Spain
Sweden
Switzerland
Thailand
Turkiye
Uganda
United Kingdom
United States of America
Uruguay
Uzbekistan
Venezuela


Tras el proceso de armonización de países, se identificaron 71 países con información disponible simultáneamente en los conjuntos de datos de GLOBOCAN, ONU, BM y DIRAC.
Ahora, se procede con la fusión de los datos por país y año, empezando con un INNER JOIN y comprobando los números de países disponibles.


In [47]:
df_merged = glob_clean.merge(
    dirac_clean,
    on="Country_harmonized",
    how="inner"
).merge(
    ONU_c,
    on=["Country_harmonized", "Year"],
    how="inner"
).merge(
    df_PIB_clean,
    on=["Country_harmonized", "Year"],
    how="inner"
)

print(df_merged['Country_harmonized'].unique())
print(len(df_merged['Country_harmonized'].unique()))
print(df_merged["Year"].unique())
print(df_merged.head())


['Belarus' 'Canada' 'Chile' 'China' 'Colombia' 'Costa Rica' 'Croatia'
 'Cuba' 'Cyprus' 'Czechia' 'Denmark' 'Ecuador' 'Estonia' 'Finland'
 'France' 'Georgia' 'Germany' 'Greece' 'Argentina' 'Guatemala' 'Guyana'
 'Hungary' 'Iceland' 'India' 'Australia' 'Ireland' 'Israel' 'Italy'
 'Japan' 'Austria' 'South Korea' 'Kuwait' 'Kyrgyzstan' 'Latvia'
 'Lithuania' 'Luxembourg' 'Malta' 'Bahrain' 'Mauritius' 'Mexico' 'Moldova'
 'Armenia' 'New Zealand' 'Nicaragua' 'Belgium' 'Norway' 'Panama'
 'Paraguay' 'Philippines' 'Poland' 'Portugal' 'Puerto Rico' 'Qatar'
 'Romania' 'Serbia' 'Singapore' 'Slovakia' 'Slovenia' 'South Africa'
 'Spain' 'Sweden' 'Switzerland' 'Brazil' 'Thailand' 'Turkiye' 'Uganda'
 'United States of America' 'Uruguay' 'Uzbekistan' 'Venezuela'
 'United Kingdom']
71
[2007 2008 2009 2010 2011 2012 2013 2014 2015 2016 2017 2018 2019 2020
 2021 2022 2023]
  Cancer label     Sex       Type  Year  ASR (World)  Crude rate  \
0       Cervix  Female  Incidence  2007    12.880989   17.994184   
1 

In [48]:
df_merged.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1456 entries, 0 to 1455
Data columns (total 21 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Cancer label        1456 non-null   object 
 1   Sex                 1456 non-null   object 
 2   Type                1456 non-null   object 
 3   Year                1456 non-null   int64  
 4   ASR (World)         1456 non-null   float64
 5   Crude rate          1456 non-null   float64
 6   Cumulative risk     1456 non-null   float64
 7   Total               1456 non-null   int64  
 8   Country_harmonized  1456 non-null   object 
 9   Country             1456 non-null   object 
 10  Region Name         1456 non-null   object 
 11  RTCenters           1456 non-null   int64  
 12  Linac               1456 non-null   int64  
 13  Protontherapy       1456 non-null   int64  
 14  XRay                1456 non-null   int64  
 15  Brachytherapy       1456 non-null   int64  
 16  Last U

In [49]:
df_merged = df_merged.drop(columns=["Country"])


In [50]:
#Comprobando que la fusión se ha realizado con éxito con los 71 paises.
print(df_merged["Country_harmonized"].nunique())


print(df_merged.head())
print(df_merged["Year"].unique())

print("Numero de duplicados:", df_merged.duplicated(
    subset=["Country_harmonized", "Cancer label", "Sex", "Type", "Year"]
).sum())

#cuantas filas han quedado con el tipo de cancer, sexo, año y pais.
df_merged.shape

71
  Cancer label     Sex       Type  Year  ASR (World)  Crude rate  \
0       Cervix  Female  Incidence  2007    12.880989   17.994184   
1       Cervix  Female  Incidence  2008    13.089796   18.763663   
2       Cervix  Female  Incidence  2009    12.789255   18.367835   
3       Cervix  Female  Incidence  2010    12.684533   18.316613   
4       Cervix  Female  Incidence  2011    13.305420   18.935117   

   Cumulative risk  Total Country_harmonized     Region Name  RTCenters  \
0         1.285189    931            Belarus  Eastern Europe         13   
1         1.296157    956            Belarus  Eastern Europe         13   
2         1.287709    934            Belarus  Eastern Europe         13   
3         1.270706    930            Belarus  Eastern Europe         13   
4         1.313381    960            Belarus  Eastern Europe         13   

   Linac  Protontherapy  XRay  Brachytherapy Last Update  Population  \
0     31              0    11             17        2024   956313

(1456, 20)

In [51]:
#cuantos NaN
df_merged.isna().sum()[df_merged.isna().sum() > 0]




,0


Guardando el dataset

In [52]:
# se guarda el dataset depurado
df_merged.to_csv(output_path / "df_merged.csv", index=False)


#fguardando 2022
df_merged_2022 = df_merged[df_merged["Year"] == 2022].copy()

# Guardar CSV
df_merged_2022.to_csv(output_path / "df_merged2022.csv", index=False)

#se separan por sexo

df_male = df_merged[df_merged["Sex"] == "Male"].copy()
df_female = df_merged[df_merged["Sex"] == "Female"].copy()

In [54]:

df_merged["Country_harmonized"].unique()


array(['Belarus', 'Canada', 'Chile', 'China', 'Colombia', 'Costa Rica',
       'Croatia', 'Cuba', 'Cyprus', 'Czechia', 'Denmark', 'Ecuador',
       'Estonia', 'Finland', 'France', 'Georgia', 'Germany', 'Greece',
       'Argentina', 'Guatemala', 'Guyana', 'Hungary', 'Iceland', 'India',
       'Australia', 'Ireland', 'Israel', 'Italy', 'Japan', 'Austria',
       'South Korea', 'Kuwait', 'Kyrgyzstan', 'Latvia', 'Lithuania',
       'Luxembourg', 'Malta', 'Bahrain', 'Mauritius', 'Mexico', 'Moldova',
       'Armenia', 'New Zealand', 'Nicaragua', 'Belgium', 'Norway',
       'Panama', 'Paraguay', 'Philippines', 'Poland', 'Portugal',
       'Puerto Rico', 'Qatar', 'Romania', 'Serbia', 'Singapore',
       'Slovakia', 'Slovenia', 'South Africa', 'Spain', 'Sweden',
       'Switzerland', 'Brazil', 'Thailand', 'Turkiye', 'Uganda',
       'United States of America', 'Uruguay', 'Uzbekistan', 'Venezuela',
       'United Kingdom'], dtype=object)

In [62]:
# Ver cuántos países hay por cada año
print("Países disponibles por año:")
print("-"*50)
year_counts = df_merged.groupby('Year')['Country_harmonized'].nunique().sort_values(ascending=False)
print(year_counts)
print(f"\n✅ Año con MÁS países: {year_counts.idxmax()} (n={year_counts.max()})")

Países disponibles por año:
--------------------------------------------------
Year
2007    71
2008    71
2009    71
2010    71
2011    71
2012    71
2013    71
2014    71
2015    71
2016    71
2017    70
2018    61
2019    60
2020    55
2021    51
2022    40
2023    14
Name: Country_harmonized, dtype: int64

✅ Año con MÁS países: 2007 (n=71)
